In [1]:
# TODO: load models and make predictions in a way where i can easily substitute in non-sklearn models/pipelines

# 1. Loading

## 1.1. Imports and prep

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import joblib
import seaborn as sns
from scipy import stats
from scipy.optimize import curve_fit
import os
import sys
from loguru import logger
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error, root_mean_squared_error
from sklearn.model_selection import KFold, TimeSeriesSplit, train_test_split, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from pygam import LinearGAM, s, l, te
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils.validation import check_is_fitted
from gplearn.genetic import SymbolicRegressor

In [3]:
WINDOW_LENGTH = "5min"

In [4]:
notebook_dir = os.getcwd()
notebook_parent_dir = os.path.dirname(notebook_dir)
notebook_parent_parent_dir = os.path.dirname(notebook_parent_dir)
train_test_dir = os.path.join(notebook_parent_parent_dir, "data", "train-and-test")
train_test_split_path = os.path.join(train_test_dir, f"train_test_splits_{WINDOW_LENGTH}.npz")

In [5]:
sys.path.append('..')  # points to the Modelling/ folder

from gam import SklearnGAM

## 1.2. Data

In [6]:
data = np.load(train_test_split_path, allow_pickle=False)
X_train = data['X_train']
X_test  = data['X_test']
y_train = data['y_train']
y_test  = data['y_test']

## 1.3. Models

In [10]:
model_pipelines = {'GAM': None, 'NN': None, 'random_forest': None}

In [11]:
for model in model_pipelines.keys():
    # find the pipeline
    model_pipeline_path = os.path.join(notebook_parent_parent_dir, "outputs", "models", f"{model}_pipeline_{WINDOW_LENGTH}.joblib")
    model_pipeline = joblib.load(model_pipeline_path)
    
    # add the pipeline to the dict
    model_pipelines[model] = model_pipeline

# 2. Assumptions & Robustness Checks

## 2.1. GAM

### 2.1.1. Model Summary

In [13]:
# 1. Extract the raw pyGAM model from our fitted pipeline
gam_pipeline = model_pipelines["GAM"]
gam_model = gam_pipeline.named_steps["model"].gam_model_
gam_model.summary()


LinearGAM                                                                                                 
=============================================== ==========================================================
Distribution:                        NormalDist Effective DoF:                                     33.3441
Link Function:                     IdentityLink Log Likelihood:                               -272875.7902
Number of Samples:                        35880 AIC:                                           545820.2686
                                                AICc:                                          545820.3363
                                                GCV:                                           236852.7642
                                                Scale:                                             486.268
                                                Pseudo R-Squared:                                   0.8945
Feature Function                  Lam

C:\Users\Augus\AppData\Local\Temp\ipykernel_44888\371537878.py:4: UserWarning: KNOWN BUG: p-values computed in this summary are likely much smaller than they should be. 
 
Please do not make inferences based on these values! 

Collaborate on a solution, and stay up to date at: 
github.com/dswah/pyGAM/issues/163 

  gam_model.summary()


In [ ]:
# 2. Generate a perfectly spaced grid of 100 points for the fouling feature
# This creates a synthetic array where ONLY the fouling feature changes.
XX = actual_pygam_model.generate_X_grid(term=days_cleaning_idx)

# 3. Extract the partial dependence (the isolated kW penalty) and the 95% confidence intervals
pdep, confi = actual_pygam_model.partial_dependence(term=days_cleaning_idx, X=XX, width=0.95)

# 3. Analysis

In [ ]:
# Make a function that at least takes the path of controlled variable as input and returns the controlled var dataframe

# 4. Sensitivity Analysis